Dcouments
DOcument Abstraction

In [9]:
from langchain_core.documents import Document

documents = [Document(
    page_content = "Dogs are great",
    metadata = {"source": "mammal-pets-doc"}
),
Document(
    page_content = "cats are great",
    metadata = {"source": "mammal-pets-doc"}

)]
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='cats are great')]

Vector Stores

In [11]:
from langchain_chroma import Chroma
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
llm = ChatGroq(groq_api_key = groq_api_key, model="meta-llama/llama-4-scout-17b-16e-instruct")


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings( model_name = "all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3794.48it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
vector_db = Chroma.from_documents(documents,embeddings)

In [14]:
vector_db.similarity_search("cat")

[Document(id='29d65bb2-d94e-4413-af44-02044b316932', metadata={'source': 'mammal-pets-doc'}, page_content='cats are great'),
 Document(id='e49d2d86-1901-4873-af47-4485a8995739', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great')]

In [15]:
## ASYNC QUERY
await vector_db.asimilarity_search("cat")

[Document(id='29d65bb2-d94e-4413-af44-02044b316932', metadata={'source': 'mammal-pets-doc'}, page_content='cats are great'),
 Document(id='e49d2d86-1901-4873-af47-4485a8995739', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great')]

In [16]:
vector_db.similarity_search_with_score('cat')

[(Document(id='29d65bb2-d94e-4413-af44-02044b316932', metadata={'source': 'mammal-pets-doc'}, page_content='cats are great'),
  0.8665671348571777),
 (Document(id='e49d2d86-1901-4873-af47-4485a8995739', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great'),
  1.3623692989349365)]

In [17]:
#Retrievers
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
retriever = RunnableLambda(vector_db.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='29d65bb2-d94e-4413-af44-02044b316932', metadata={'source': 'mammal-pets-doc'}, page_content='cats are great')],
 [Document(id='e49d2d86-1901-4873-af47-4485a8995739', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great')]]

In [19]:
##implement as retriever method
retriever =vector_db.as_retriever(
    search_type = "similarity",
    search_kwargs= {"k":1}
)

retriever.batch(["cat","dog"])

[[Document(id='29d65bb2-d94e-4413-af44-02044b316932', metadata={'source': 'mammal-pets-doc'}, page_content='cats are great')],
 [Document(id='e49d2d86-1901-4873-af47-4485a8995739', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great')]]